# Batch Scoring — Employee Attrition Risk Predictions

**Purpose:** Load the trained Logistic Regression model from MLflow, score all 1,470 employees,
and write risk predictions to `workspace.gold.gold_attrition_predictions`.

This closes the loop between model training and business consumption:

```
gold_ml_features → Trained Model → Predictions Table → Power BI
```

### Pipeline
1. Retrieve best model + fitted scaler from MLflow
2. Load feature data from `workspace.gold.gold_ml_features`
3. Score all employees → attrition probability
4. Assign risk tiers (High / Medium / Low)
5. Write predictions to Gold Delta table

### Output
- **Table:** `workspace.gold.gold_attrition_predictions`
- **Grain:** 1 row per employee (1,470 rows)
- **Key columns:** `attrition_probability`, `risk_tier`, `predicted_attrition`

> **Note:** This scores the full dataset (train + test). Probabilities are calibrated
> from the model trained on 80% of the data. Test-set metrics (AUC 0.84, Recall 0.68)
> remain the unbiased performance estimate.

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib
from datetime import datetime

print(f"Scoring started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Retrieve Best Model from MLflow

In [ ]:
EXPERIMENT_NAME = "/Users/mahmoudgribej7@gmail.com/employee-attrition-ibm-hr"

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
assert experiment is not None, f"Experiment '{EXPERIMENT_NAME}' not found. Run training notebook first."

# Find the BEST run (logged with BEST_ prefix during training)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.run_name LIKE 'BEST_%'",
    order_by=["metrics.auc_roc DESC"],
    max_results=1
)

if len(runs) == 0:
    # Fallback: pick the run with highest AUC-ROC
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.auc_roc DESC"],
        max_results=1
    )

best_run_id = runs.iloc[0]["run_id"]
best_auc = runs.iloc[0]["metrics.auc_roc"]
best_name = runs.iloc[0]["tags.mlflow.runName"]

print(f"Best run:  {best_name}")
print(f"Run ID:    {best_run_id}")
print(f"AUC-ROC:   {best_auc:.4f}")

## 2. Load Model & Scaler Artifacts

In [ ]:
# Load trained model
model_uri = f"runs:/{best_run_id}/model"
model = mlflow.sklearn.load_model(model_uri)
print(f"Model loaded: {type(model).__name__}")
print(f"  Params: C={model.C}, penalty={model.penalty}, solver={model.solver}")

# Load fitted scaler (saved as joblib artifact)
scaler_artifact = mlflow.artifacts.download_artifacts(f"runs:/{best_run_id}/scaler/scaler.joblib")
scaler = joblib.load(scaler_artifact)
print(f"Scaler loaded: {type(scaler).__name__} ({scaler.n_features_in_} features)")

# Load feature column names (saved during training)
feature_cols_artifact = mlflow.artifacts.download_artifacts(f"runs:/{best_run_id}/feature_columns.txt")
with open(feature_cols_artifact, 'r') as f:
    feature_cols = f.read().strip().split(',')
print(f"Feature columns: {len(feature_cols)}")

## 3. Load Feature Data

In [ ]:
df = spark.table("workspace.gold.gold_ml_features").toPandas()
print(f"Loaded: {df.shape[0]:,} employees x {df.shape[1]} columns")
print(f"Attrition distribution: {df['attrition'].value_counts().to_dict()}")

## 4. Score All Employees

In [ ]:
# Prepare feature matrix (same pipeline as training)
X = df[feature_cols].fillna(0)
X_scaled = scaler.transform(X)

# Predict
probabilities = model.predict_proba(X_scaled)[:, 1]  # P(attrition = 1)
predictions = model.predict(X_scaled)

print(f"Scored {len(probabilities):,} employees")
print(f"Probability range: [{probabilities.min():.4f}, {probabilities.max():.4f}]")
print(f"Mean probability:  {probabilities.mean():.4f}")
print(f"Predicted to leave: {predictions.sum():,} / {len(predictions):,}")

## 5. Build Predictions Table

In [ ]:
# ---- Reconstruct human-readable labels from one-hot columns ----

# Department
df['department_name'] = np.where(
    df['dept_sales'] == 1, 'Sales',
    np.where(df['dept_research_dev'] == 1, 'Research & Development', 'Human Resources')
)

# Job role
role_map = {
    'role_sales_executive': 'Sales Executive',
    'role_research_scientist': 'Research Scientist',
    'role_lab_technician': 'Laboratory Technician',
    'role_manufacturing_director': 'Manufacturing Director',
    'role_healthcare_rep': 'Healthcare Representative',
    'role_manager': 'Manager',
    'role_sales_rep': 'Sales Representative',
    'role_research_director': 'Research Director',
    'role_hr': 'Human Resources',
}
df['job_role_name'] = 'Unknown'
for col, name in role_map.items():
    df.loc[df[col] == 1, 'job_role_name'] = name

# Marital status
df['marital_status_name'] = np.where(
    df['marital_single'] == 1, 'Single',
    np.where(df['marital_married'] == 1, 'Married', 'Divorced')
)

# Other labels
df['overtime_label'] = df['overtime'].map({1: 'Yes', 0: 'No'})
df['gender_label'] = df['gender'].map({1: 'Male', 0: 'Female'})

# ---- Attach predictions ----
df['attrition_probability'] = np.round(probabilities, 4)
df['predicted_attrition'] = predictions.astype(int)

# Risk tiers based on probability thresholds
df['risk_tier'] = pd.cut(
    df['attrition_probability'],
    bins=[-0.01, 0.30, 0.60, 1.01],
    labels=['Low', 'Medium', 'High']
)

df['attrition_label'] = df['attrition'].map({1: 'Yes', 0: 'No'})

print("Labels reconstructed, predictions attached.")

In [ ]:
# Build final output table with selected columns
output_df = df[[
    # Identifiers & demographics
    'department_name', 'job_role_name', 'age', 'gender_label',
    'marital_status_name', 'distance_from_home',
    # Job characteristics
    'job_level', 'overtime_label', 'monthly_income',
    'years_at_company', 'total_working_years',
    'years_in_current_role', 'years_since_last_promotion',
    # Satisfaction scores
    'environment_satisfaction', 'job_satisfaction',
    'work_life_balance', 'relationship_satisfaction',
    # Actual outcome
    'attrition', 'attrition_label',
    # Model predictions
    'attrition_probability', 'predicted_attrition', 'risk_tier'
]].copy()

# Deterministic sort and key assignment
output_df = output_df.sort_values(
    ['department_name', 'job_role_name', 'age', 'monthly_income']
).reset_index(drop=True)
output_df.insert(0, 'employee_key', range(1, len(output_df) + 1))

print(f"Output table: {output_df.shape[0]:,} rows x {output_df.shape[1]} columns")
output_df.head(10)

## 6. Write to Gold Delta Table

In [ ]:
# Convert risk_tier from Categorical to string (Spark doesn't handle pd.Categorical)
output_df['risk_tier'] = output_df['risk_tier'].astype(str)

# Write to Delta
output_spark_df = spark.createDataFrame(output_df)
output_spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.gold_attrition_predictions")

# Verify
verify_count = spark.table("workspace.gold.gold_attrition_predictions").count()
print(f"Written to workspace.gold.gold_attrition_predictions")
print(f"Verified: {verify_count:,} rows")

## 7. Scoring Results & Risk Analysis

In [ ]:
print("=" * 65)
print("  RISK TIER DISTRIBUTION")
print("=" * 65)
risk_dist = output_df['risk_tier'].value_counts()
for tier in ['High', 'Medium', 'Low']:
    count = risk_dist.get(tier, 0)
    pct = count / len(output_df) * 100
    bar = '█' * int(pct / 2)
    print(f"  {tier:8s}  {count:5,}  ({pct:5.1f}%)  {bar}")

print(f"\n{'=' * 65}")
print("  RISK BY DEPARTMENT")
print("=" * 65)
dept_risk = output_df.groupby('department_name').agg(
    avg_risk=('attrition_probability', 'mean'),
    high_risk_count=('risk_tier', lambda x: (x == 'High').sum()),
    headcount=('employee_key', 'count')
).sort_values('avg_risk', ascending=False)

for dept, row in dept_risk.iterrows():
    print(f"  {dept:28s}  Avg Risk: {row['avg_risk']:.1%}  "
          f"High-risk: {int(row['high_risk_count'])}/{int(row['headcount'])}")

print(f"\n{'=' * 65}")
print("  RISK BY JOB ROLE (Top 5)")
print("=" * 65)
role_risk = output_df.groupby('job_role_name')['attrition_probability'].mean().sort_values(ascending=False).head(5)
for role, risk in role_risk.items():
    print(f"  {role:30s}  {risk:.1%}")

In [ ]:
print("=" * 65)
print("  TOP 15 HIGHEST-RISK EMPLOYEES")
print("=" * 65)
top = output_df.nlargest(15, 'attrition_probability')
for i, (_, row) in enumerate(top.iterrows(), 1):
    actual = "LEFT" if row['attrition'] == 1 else "stayed"
    print(
        f"  {i:2d}. [{row['risk_tier']:6s}] "
        f"P={row['attrition_probability']:.1%}  "
        f"{row['department_name']:25s} "
        f"{row['job_role_name']:25s} "
        f"Age {int(row['age']):2d}  "
        f"${int(row['monthly_income']):>6,}/mo  "
        f"({actual})"
    )

In [ ]:
print("=" * 65)
print("  MODEL VALIDATION (full dataset)")
print("=" * 65)

actual_pos = int(output_df['attrition'].sum())
actual_neg = int((output_df['attrition'] == 0).sum())
pred_pos = int(output_df['predicted_attrition'].sum())
tp = int(((output_df['attrition'] == 1) & (output_df['predicted_attrition'] == 1)).sum())
fp = int(((output_df['attrition'] == 0) & (output_df['predicted_attrition'] == 1)).sum())
fn = int(((output_df['attrition'] == 1) & (output_df['predicted_attrition'] == 0)).sum())
tn = int(((output_df['attrition'] == 0) & (output_df['predicted_attrition'] == 0)).sum())

print(f"  Actual who left:       {actual_pos:,}")
print(f"  Actual who stayed:     {actual_neg:,}")
print(f"  Predicted to leave:    {pred_pos:,}")
print(f"")
print(f"  True Positives:        {tp}  (correctly flagged as at-risk)")
print(f"  False Positives:       {fp}  (false alarms)")
print(f"  False Negatives:       {fn}  (missed attritions)")
print(f"  True Negatives:        {tn}  (correctly identified as stable)")
print(f"")
recall = tp / actual_pos if actual_pos > 0 else 0
precision = tp / pred_pos if pred_pos > 0 else 0
print(f"  Recall (sensitivity):  {recall:.1%}  — catches {recall:.0%} of actual leavers")
print(f"  Precision:             {precision:.1%}  — {precision:.0%} of flagged employees actually left")
print(f"")
print(f"  Note: These are full-dataset metrics (train+test). The unbiased")
print(f"  test-set performance is AUC-ROC=0.84, Recall=0.68, F1=0.49.")

In [ ]:
# Risk tier vs actual attrition crosstab
print("=" * 65)
print("  RISK TIER vs ACTUAL ATTRITION")
print("=" * 65)

cross = output_df.groupby('risk_tier').agg(
    total=('attrition', 'count'),
    actual_left=('attrition', 'sum'),
    avg_probability=('attrition_probability', 'mean')
).reset_index()
cross['actual_attrition_rate'] = cross['actual_left'] / cross['total']

for _, row in cross.iterrows():
    print(
        f"  {row['risk_tier']:8s}  "
        f"n={int(row['total']):>5,}  "
        f"Actually left: {int(row['actual_left']):>3}  "
        f"Actual rate: {row['actual_attrition_rate']:.1%}  "
        f"Avg P: {row['avg_probability']:.1%}"
    )

print(f"\nScoring complete. Table ready for Power BI consumption.")

## Power BI Integration

The `gold_attrition_predictions` table is now available in Databricks SQL.  

**In Power BI:**
1. Add `workspace.gold.gold_attrition_predictions` via Get Data → Databricks
2. Key columns for visuals:
   - `attrition_probability` — continuous risk score (0–1)
   - `risk_tier` — categorical (High / Medium / Low) for slicers and color coding
   - `predicted_attrition` — binary flag for counts
3. Suggested visuals:
   - **KPI card:** High-risk employee count
   - **Bar chart:** Risk tier distribution
   - **Table:** Top at-risk employees with department, role, probability
   - **Scatter:** Probability vs Monthly Income, colored by risk tier